<a href="https://colab.research.google.com/github/humaaslam46/flyRank-ml-Internship-tasks/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humaaslam46/Internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("Internship-ml"):
        os.system("git clone --depth 1 https://github.com/humaaslam46/Internship-ml")
    os.chdir("Internship-ml")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
import pandas as pd, numpy as np, json
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
features = ["impressions_90d","days_since_last_update","avg_position","ctr","word_count","engagement_rate","content_age_days","search_volume"]
print("Ready:", df.shape)

Ready: (30000, 45)


## 1. Ranked actions + reason codes

Four archetypes, each mapped to one action, ranked by the model's decline probability:

- HIGH_VISIBILITY_DECLINE_RISK -> "Refresh now - highest priority" (high traffic + high predicted decline risk)
- MODERATE_REVIEW -> "Review this cycle" (mid-tier visibility/risk, not urgent but worth checking)
- HIGH_VISIBILITY_STABLE -> "Protect / leave as-is" (high traffic, low predicted risk - don't touch a working page)
- LOW_TRAFFIC_MONITOR_ONLY -> "Monitor only, not enough visibility for a confident call" (bottom 25% by impressions - this is exactly where ML-08/ML-09 showed the model's errors concentrate, so I deliberately route these away from confident action)

In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

Xtr = train[features].replace([np.inf,-np.inf],np.nan).fillna(0)
Xte = test[features].replace([np.inf,-np.inf],np.nan).fillna(0)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(Xtr, train["is_declining"])
test["decline_prob"] = rf.predict_proba(Xte)[:,1]

vis_median = test["impressions_90d"].median()
low_vis_thresh = test["impressions_90d"].quantile(0.25)

def archetype(row):
    high_vis = row["impressions_90d"] >= vis_median
    high_risk = row["decline_prob"] >= 0.6
    low_vis = row["impressions_90d"] < low_vis_thresh
    if low_vis:
        return "LOW_TRAFFIC_MONITOR_ONLY", "Monitor only - not enough visibility for a confident call"
    if high_vis and high_risk:
        return "HIGH_VISIBILITY_DECLINE_RISK", "Refresh now - highest priority"
    if high_vis and not high_risk:
        return "HIGH_VISIBILITY_STABLE", "Protect / leave as-is"
    return "MODERATE_REVIEW", "Review this cycle"

results = test.apply(lambda r: archetype(r), axis=1)
test["reason_code"] = [r[0] for r in results]
test["action"] = [r[1] for r in results]
queue = test.sort_values("decline_prob", ascending=False)

print(queue.head(10)[["content_id","impressions_90d","decline_prob","reason_code","action"]].to_string(index=False))
print("\nAction distribution:")
print(test["action"].value_counts())

          content_id  impressions_90d  decline_prob                  reason_code                         action
content_f55fd2d8ed04             2237      0.806006 HIGH_VISIBILITY_DECLINE_RISK Refresh now - highest priority
content_2a228ce7aa1b              155      0.799823              MODERATE_REVIEW              Review this cycle
content_2ba626fea4d6              360      0.795178 HIGH_VISIBILITY_DECLINE_RISK Refresh now - highest priority
content_884c401ce126              101      0.791808              MODERATE_REVIEW              Review this cycle
content_7e24f3b27d73              825      0.791529 HIGH_VISIBILITY_DECLINE_RISK Refresh now - highest priority
content_4fc70e470460              766      0.791510 HIGH_VISIBILITY_DECLINE_RISK Refresh now - highest priority
content_7766ffacdcfa              657      0.790676 HIGH_VISIBILITY_DECLINE_RISK Refresh now - highest priority
content_700de55b1459             1258      0.790632 HIGH_VISIBILITY_DECLINE_RISK Refresh now - highest p

## 2. Intended use and limits

Intended use: a content strategist reviews the "Refresh now" and "Review this cycle"
tiers each cycle, in that order, as a prioritized starting point - not a final answer.

Limits: this stops being valid the moment a page's underlying signals shift
meaningfully (a redesign, a big traffic event, a client campaign) since the
model was trained on a snapshot. It's also weakest exactly on low-traffic
pages (see LOW_TRAFFIC_MONITOR_ONLY) - confirmed directly in ML-09's error
audit, where both false positives and false negatives clustered there. The
label itself is a current-window proxy, not a verified future outcome, so
this ranks likely-worth-reviewing pages - it does not predict traffic with
certainty.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

A human must confirm before acting: whether a "Refresh now" page's decline
is actually content-related, versus a seasonal dip, a tracking gap, or a
known business change (e.g. a product discontinued) - the model has no way
to distinguish these.

Must NEVER be automated: actually editing, merging, or removing a page
based on this queue alone. This playbook produces a reviewed starting list,
never a direct-action pipeline - especially since the "decline" label is a
proxy, not a confirmed outcome, and acting on a wrong call at scale (e.g.
auto-merging pages) is not reversible the way skipping a review cycle is.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

Retrain if: the "Refresh now" tier's real-world precision (checked manually
on a sample each cycle) drops meaningfully below the 0.540 Precision@50
measured here - that's a sign the underlying patterns have drifted from
what the model learned.

Also retrain if: the action distribution shifts sharply cycle to cycle
(e.g. "Refresh now" suddenly balloons from ~16% to 40%+ of pages) - that's
more likely a data or pipeline issue than a genuine spike in declining
content.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)

metrics = {
    "baseline_p50": 0.380, "rf_p50_grouped": 0.540, "improvement_x": 1.42,
    "rf_p50_naive_leaked": 0.860, "leakage_inflation_pct": 59.3,
    "n_test_rows": len(test), "n_test_clients": int(test["client_id"].nunique()),
    "action_distribution": test["action"].value_counts().to_dict()
}
with open("work/outputs/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported queue and metrics.json to work/outputs/")

Exported queue and metrics.json to work/outputs/


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.